In [27]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

PLL_DIR = Path.cwd().resolve().parents[2] / "results" / "pll"  # notebook lives in sw/tools/notebooks/
RESULT_CSV = None  # e.g. PLL_DIR / "pll_spectrum_div_S5_20261005_170000.csv"; None = newest run

if RESULT_CSV is None:
    RESULT_CSV = sorted((p for p in PLL_DIR.glob("pll_spectrum_div_*.csv") if not p.stem.endswith("_traces")),
                        key=lambda p: p.stat().st_mtime)[-1]
RESULT_CSV = Path(RESULT_CSV)
TRACE_CSV = RESULT_CSV.with_name(RESULT_CSV.stem + "_traces.csv")
divs = pd.read_csv(RESULT_CSV).sort_values(["total_div", "pll_clk_o_en"]).reset_index(drop=True)
traces = pd.read_csv(TRACE_CSV) if TRACE_CSV.exists() else pd.DataFrame(columns=["divider", "kind"])
divs["path"] = np.where(divs["pll_clk_o_en"] == 1, "inside the PLL", "outside the PLL")
# PLL output and external divider from the divider settings (runs before these columns were written lack them)
divs["pll_div"] = 2.0 ** divs["set_div_freq"]
divs["ext_div"] = divs["total_div"] / divs["pll_div"]
divs["f_pll_out"] = divs["f_vco"] / divs["pll_div"]
divs["n_meas"] = divs["f_vco"] / divs["f_out_meas"]  # f_vco / carrier on the pad: should equal total_div
divs["divider_ok"] = (divs["n_meas"] / divs["total_div"] - 1).abs() < 1e-3
ok = divs[divs["status"] == "ok"]
r0 = divs.iloc[0]
print(f"{RESULT_CSV.name}: {len(divs)} dividers, {(divs['locked'] == 1).sum()} locked, {len(ok)} measured ok; "
      f"f_vco {r0['f_vco'] / 1e6:g} MHz, c{r0['vco_tune_coarse']:.0f} min {r0['vco_current_min']:.0f} "
      f"max {r0['vco_current_max']:.0f}, |Kvco| {r0['kvco_pred'] / 1e6:.0f} MHz/V; common band {r0['common_band']} Hz")
no_carrier = divs[divs["status"] == "no_carrier"]
if len(no_carrier):
    print("no carrier on the pad (not measured): " + ", ".join(
        f"{r.divider} ({r.f_out / 1e6:g} MHz on the pad, {r.carrier_dbm:.1f} dBm)" for r in no_carrier.itertuples()))

pd.DataFrame({
    "divider": divs["divider"],
    "N": divs["total_div"],
    "PLL output [MHz]": divs["f_pll_out"] / 1e6,
    "ext div": divs["ext_div"],
    "pad f [MHz]": divs["f_out"] / 1e6,
    "pad f meas [MHz]": divs["f_out_meas"] / 1e6,
    "N measured": divs["n_meas"],
    "divider ok": divs["divider_ok"],
    "carrier on the pad [dBm]": divs["carrier_dbm"],
    "band up to [Hz]": divs["offset_max"],
    "jitter common, markers [ps]": divs["time_jitter_common_ps"],
    "jitter common, curve [ps]": divs["time_jitter_trace_common_ps"],
    "jitter full band, curve [ps]": divs["time_jitter_trace_ps"],
    "jitter full band, spurs [ps]": divs["time_jitter_spur_ps"],
    "locked": divs["locked"],
    "status": divs["status"],
}).round(3)

pll_spectrum_div_S5_20261005_174743.csv: 9 dividers, 9 locked, 9 measured ok; f_vco 1000 MHz, c2 min 11 max 4, |Kvco| 1146 MHz/V; common band 1000-1e+06 Hz


,divider,N,PLL output [MHz],ext div,pad f [MHz],pad f meas [MHz],N measured,divider ok,carrier on the pad [dBm],band up to [Hz],"jitter common, markers [ps]","jitter common, curve [ps]","jitter full band, curve [ps]","jitter full band, spurs [ps]",locked,status
0,ext /2 x /4,8,500.000,4.0,125.000,125.000,8.0,True,-3.944,50000000.0,734.873,1267.726,1703.893,633.130,1,ok
1,int /8,8,125.000,1.0,125.000,125.000,8.0,True,-3.952,50000000.0,1009.580,1290.927,1740.634,649.770,1,ok
2,ext /1 x /10,10,1000.000,10.0,100.000,100.000,10.0,True,-2.235,10000000.0,685.585,1182.345,1521.870,610.230,1,ok
3,int /16,16,62.500,1.0,62.500,62.500,16.0,True,0.613,10000000.0,690.166,1101.392,1392.737,557.163,1,ok
4,int /32,32,31.250,1.0,31.250,31.250,32.0,True,-1.840,10000000.0,526.430,1027.117,1275.548,556.863,1,ok
5,ext /4 x /10,40,250.000,10.0,25.000,25.000,40.0,True,-2.288,10000000.0,592.420,1112.363,1355.108,581.266,1,ok
6,int /64,64,15.625,1.0,15.625,15.625,64.0,True,-3.087,1000000.0,500.236,967.263,967.263,436.575,1,ok
7,ext /1 x /100,100,1000.000,100.0,10.000,10.000,100.0,True,-4.002,1000000.0,461.650,1075.334,1075.334,593.554,1,ok
8,int /128,128,7.812,1.0,7.812,7.812,128.0,True,-4.540,1000000.0,496.095,1008.242,1008.242,461.865,1,ok


In [29]:
INK, GRID = "#52514e", "#e4e3df"
PATH_COLORS = {"inside the PLL": "#2a78d6", "outside the PLL": "#eb6834"}
REF_TXT = f"reference {divs['f_ref'].iloc[0] / 1e6:.4g} MHz"  # input of the PLL, the same for every divider
OFFSET_AXIS = dict(type="log", tickvals=[1e3, 1e4, 1e5, 1e6, 1e7, 1e8],
                   ticktext=["1 kHz", "10 kHz", "100 kHz", "1 MHz", "10 MHz", "100 MHz"],
                   minor=dict(showgrid=True, gridcolor="#f1f0ed"))
FREQ_AXIS = dict(type="log", tickvals=[1, 10, 100, 1e3], ticktext=["1 MHz", "10 MHz", "100 MHz", "1 GHz"],
                 minor=dict(showgrid=True, gridcolor="#f1f0ed"))


def save(fig, name):
    """Interactive HTML and a static PNG (kaleido) next to the CSV, then show the figure."""
    base = RESULT_CSV.with_name(f"{RESULT_CSV.stem}_{name}")
    fig.write_html(base.with_suffix(".html"))
    try:
        fig.write_image(base.with_suffix(".png"), width=1100, height=fig.layout.height or 600, scale=2)
        print(f"saved {base}.html and .png")
    except Exception as e:  # kaleido >= 1.0 needs a Chrome/Chromium install (`plotly_get_chrome`)
        print(f"saved {base}.html; PNG export failed: {e}")
    fig.show()

In [30]:
# Divider check: N measured (f_vco / carrier on the pad) against the set total divider; on the line when right.
# (A digital divider is exact: this only catches a wrong N, not a precision error.)
fig = go.Figure()
lim = [divs["total_div"].min() / 1.5, divs["total_div"].max() * 1.5]
fig.add_scatter(x=lim, y=lim, mode="lines", line=dict(color=INK, width=1, dash="dash"), hoverinfo="skip",
                name="N measured = N set")
for path, d in divs[divs["n_meas"].notna()].groupby("path"):
    fig.add_scatter(x=d["total_div"], y=d["n_meas"], mode="markers", name=path,
                    marker=dict(size=11, color=PATH_COLORS[path], line=dict(color="white", width=1)),
                    customdata=np.c_[d["divider"], d["f_out_meas"] / 1e6, d["divider_ok"]],
                    hovertemplate="%{customdata[0]}: N set %{x}, measured %{y:.4f} (carrier on the pad "
                                  "%{customdata[1]:.6f} MHz), ok %{customdata[2]}<extra></extra>")
fig.update_layout(title=f"Divider check: N measured (f_vco / carrier on the pad) vs N set, {REF_TXT}", template="plotly_white",
                  height=480, legend=dict(orientation="h", y=-0.2))
fig.update_xaxes(title="total divider set", type="log", gridcolor=GRID)
fig.update_yaxes(title="N measured", type="log", gridcolor=GRID)
save(fig, "divider_check")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_spectrum_div_S5_20261005_174743_divider_check.html and .png


In [31]:
# Phase noise referred to the VCO (pad + 20 log10 N) for every divider: ideally the curves coincide.
# Colour = total divider N (settings with the same N share a colour); solid = divider inside the PLL only (pad = PLL
# output), dotted = PLL divider + external divider. Legend: division inside the PLL x outside = total, frequencies.
from plotly.colors import sample_colorscale

PALETTE_N = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
totals = sorted(ok["total_div"].unique())
N_COLORS = dict(zip(totals, PALETTE_N if len(totals) <= len(PALETTE_N) else
                    sample_colorscale("Viridis", list(np.linspace(0, 0.9, len(totals))))))
fig = go.Figure()
for r in ok.sort_values(["total_div", "ext_div"]).itertuples():
    lt = traces[(traces["divider"] == r.divider) & (traces["kind"] == "L_trace")]
    if lt.empty:
        continue
    external = r.ext_div > 1
    name = (f"/{r.pll_div:.0f} in the PLL x /{r.ext_div:.0f} external = /{r.total_div:.0f}: "
            f"PLL output {r.f_pll_out / 1e6:g} MHz, pad {r.f_out / 1e6:g} MHz, carrier {r.carrier_dbm:.1f} dBm")
    fig.add_scatter(x=lt["freq"], y=lt["level_dbm"] + 20 * np.log10(r.total_div), mode="lines", name=name,
                    line=dict(color=N_COLORS[r.total_div], width=2.5 if external else 2,
                              dash="dot" if external else "solid"),
                    hovertemplate=f"/{r.pll_div:.0f} x /{r.ext_div:.0f}<br>%{{x:.3s}}Hz: %{{y:.1f}} dBc/Hz<extra></extra>")
fig.update_layout(title=f"Phase noise referred to the VCO (pad + 20 log10 N) per divider (should coincide), {REF_TXT}"
                        "<br><sup>solid: divider inside the PLL only, dotted: + external divider</sup>",
                  template="plotly_white", height=650, legend=dict(orientation="h", y=-0.2, font=dict(size=10)),
                  hoverlabel=dict(namelength=-1))
fig.update_xaxes(title="offset from the carrier", gridcolor=GRID, **OFFSET_AXIS)
fig.update_yaxes(title="L(f) referred to the VCO [dBc/Hz]", gridcolor=GRID)
save(fig, "phase_noise")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_spectrum_div_S5_20261005_174743_phase_noise.html and .png


In [32]:
# RMS jitter over the common band per divider (markers and full curve): should be the same for every divider.
band = ok["common_band"].iloc[0] if len(ok) else ""
fig = go.Figure()
for col, name, color in (("time_jitter_trace_common_ps", "full curve", "#2a78d6"),
                         ("time_jitter_common_ps", "markers", "#eb6834")):
    fig.add_bar(x=ok["divider"], y=ok[col], name=name, marker_color=color,
                customdata=np.c_[ok["total_div"], ok["f_out"] / 1e6, ok["carrier_dbm"]],
                hovertemplate=(f"{name}: %{{y:.1f}} ps<br>%{{x}} (/%{{customdata[0]:.0f}}, %{{customdata[1]:g}} MHz, "
                               "carrier %{customdata[2]:.1f} dBm)<extra></extra>"))
fig.update_layout(title=f"RMS time jitter over the common band ({band} Hz) per divider (same at pad, PLL output, VCO), {REF_TXT}", template="plotly_white",
                  height=470, barmode="group", legend=dict(orientation="h", y=-0.3))
fig.update_xaxes(title="divider", categoryorder="array", categoryarray=list(ok["divider"]), gridcolor=GRID)
fig.update_yaxes(title="RMS time jitter [ps]", gridcolor=GRID, rangemode="tozero")
save(fig, "jitter_common")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_spectrum_div_S5_20261005_174743_jitter_common.html and .png


In [33]:
# Carrier level on the pad vs pad frequency: how well the pad drives each frequency. Colour = path.
fig = go.Figure()
for path, d in divs[divs["carrier_dbm"].notna()].groupby("path"):
    fig.add_scatter(x=d["f_out"] / 1e6, y=d["carrier_dbm"], mode="markers", name=path,
                    marker=dict(size=11, color=PATH_COLORS[path], line=dict(color="white", width=1)),
                    customdata=np.c_[d["divider"], d["total_div"], d["f_pll_out"] / 1e6],
                    hovertemplate=("%{customdata[0]} (/%{customdata[1]}): pad %{x:g} MHz, %{y:.1f} dBm "
                                   "(PLL output %{customdata[2]} MHz)<extra></extra>"))
fig.update_layout(title=f"Carrier level on the pad vs pad frequency, {REF_TXT}", template="plotly_white", height=450,
                  legend=dict(orientation="h", y=-0.25))
fig.update_xaxes(title="pad frequency (after the external divider)", gridcolor=GRID, **FREQ_AXIS)
fig.update_yaxes(title="carrier [dBm]", gridcolor=GRID)
save(fig, "carrier_vs_f_out")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_spectrum_div_S5_20261005_174743_carrier_vs_f_out.html and .png
